In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import librosa
import cv2

from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoModel, AutoTokenizer
from torchvision.models.video import r2plus1d_18

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# =====================================================
# PATHS
# =====================================================

muril_path  = "/kaggle/input/datasets/zzy990106/murilbasecased"
EXCEL_PATH  = "/kaggle/input/datasets/harrypotter14/capstone/capstone/metadata.xlsx"
AUDIO_DIR   = "/kaggle/input/datasets/harrypotter14/capstone/capstone/audio"
VIDEO_DIR   = "/kaggle/input/datasets/harrypotter14/capstone/capstone/video"

MAX_AUDIO_LEN = 150
MAX_FRAMES    = 8


# =====================================================
# DATASET
# =====================================================

class MultiModalDataset(Dataset):


    def __init__(self, excel_path, audio_dir, video_dir, tokenizer):
        self.df        = pd.read_excel(excel_path)
        self.audio_dir = audio_dir
        self.video_dir = video_dir
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.df)

    def _find_audio(self, file_id):
        for ext in [".wav", ".mp3", ".flac", ".ogg"]:
            p = f"{self.audio_dir}/{file_id}{ext}"
            if os.path.exists(p):
                return p
        return None

    def _mask_tokens(self, ids, mask_prob=0.15):
        
        ids = ids.clone()
        real = ids != 0   # non-padding positions
        mask = real & (torch.rand_like(ids.float()) < mask_prob)
        ids[mask] = 103   # [MASK] token id
        return ids

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        file_id = str(row["id"]).strip()

        enc_l = self.tokenizer(
            str(row["text"]),
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        ids_l  = enc_l["input_ids"].squeeze(0)       # (128,)
        mask_l = enc_l["attention_mask"].squeeze(0)  # (128,)

        ids_m  = self._mask_tokens(ids_l)          
        mask_m = mask_l.clone()


        audio_path = self._find_audio(file_id)
        if audio_path is None:
            y, sr = np.zeros(16000), 16000
        else:
            y, sr = librosa.load(audio_path, sr=16000)

        mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80)
        mel = librosa.power_to_db(mel)
        mel = torch.tensor(mel.T).float()            # (T, 80)

        if mel.size(0) < MAX_AUDIO_LEN:
            pad = torch.zeros(MAX_AUDIO_LEN - mel.size(0), 80)
            mel = torch.cat([mel, pad], dim=0)
        else:
            mel = mel[:MAX_AUDIO_LEN]

        audio_len = mel.size(0)

       
        video_path = f"{self.video_dir}/{file_id}.mp4"
        cap         = cv2.VideoCapture(video_path)
        frames      = []
        total       = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        idxs        = np.linspace(0, max(total - 1, 0), MAX_FRAMES, dtype=int)

        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if ret:
                frame = cv2.resize(frame, (112, 112))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
        cap.release()

        while len(frames) < MAX_FRAMES:
            frames.append(np.zeros((112, 112, 3), dtype=np.uint8))

        video_tensor = (
            torch.tensor(np.stack(frames))          # (T, H, W, C)
            .permute(0, 3, 1, 2)                    # (T, C, H, W)
            .float() / 255.0
        )

        return {
            "audio":      mel,
            "video":      video_tensor,
            "text_ids_l": ids_l,
            "mask_l":     mask_l,
            "text_ids_m": ids_m,
            "mask_m":     mask_m,
            "label":      torch.tensor(row["label"], dtype=torch.long),
            "audio_len":  torch.tensor(audio_len),
            "video_len":  torch.tensor(MAX_FRAMES),
            "text_len":   torch.tensor(mask_l.sum()),
        }


# =====================================================
# EMBEDDING LAYERS
# =====================================================

class AcousticEmbedding(nn.Module):
    """Projects mel-spectrogram frames to d_model=256."""
    def __init__(self, n_mels=80, d_model=256):
        super().__init__()
        self.proj = nn.Linear(n_mels, d_model)

    def forward(self, x):                    # x: (B, T, 80)
        return F.relu(self.proj(x))          # (B, T, 256)


class VisualEmbedding(nn.Module):
   
    def __init__(self, d_model=256):
        super().__init__()
        backbone       = r2plus1d_18(weights=None)
        self.features  = nn.Sequential(*list(backbone.children())[:-1])
        self.proj      = nn.Linear(512, d_model)

    def forward(self, video):                # video: (B, T, C, H, W)
        B, T, C, H, W = video.shape
        video = video.permute(0, 2, 1, 3, 4) # (B, C, T, H, W)
        outs  = []
        for t in range(T):
            clip = video[:, :, t:t+1, :, :].expand(-1, -1, 3, -1, -1)
            f    = self.features(clip).flatten(1)  # (B, 512)
            outs.append(self.proj(f))              # (B, 256)
        return torch.stack(outs, dim=1)            # (B, T, 256)


class TextEmbedding(nn.Module):
    """Shared BERT encoder used for both labeled and masked streams."""
    def __init__(self, path, d_model=256):
        super().__init__()
        self.bert = AutoModel.from_pretrained(path)
        self.proj = nn.Linear(768, d_model)

    def forward(self, ids, mask):           
        out = self.bert(ids, attention_mask=mask)
        return self.proj(out.last_hidden_state)  # (B, L, 256)


# =====================================================
# CTC ALIGNMENT MODULE
# =====================================================

class CTCAlign(nn.Module):
   
    def __init__(self):
        super().__init__()
        self.ctc = nn.CTCLoss(blank=0, zero_infinity=True)

    def _interp(self, seq, target_len):
        # seq: (B, T, D) → (B, target_len, D)
        return F.interpolate(
            seq.permute(0, 2, 1),            # (B, D, T)
            size=target_len,
            mode="linear",
            align_corners=False
        ).permute(0, 2, 1)                   # (B, target_len, D)

    def forward(self, Ea, Ev, Etl, a_len, v_len, t_len):
        targets = torch.argmax(Etl, dim=-1)  # (B, L) pseudo-targets

        log_a = F.log_softmax(Ea, dim=-1).transpose(0, 1)  # (T, B, D)
        log_v = F.log_softmax(Ev, dim=-1).transpose(0, 1)

        loss_a = self.ctc(log_a, targets, a_len, t_len)
        loss_v = self.ctc(log_v, targets, v_len, t_len)

        Ea = self._interp(Ea, Etl.size(1))
        Ev = self._interp(Ev, Etl.size(1))

        return Ea, Ev, Etl, loss_a + loss_v


# =====================================================
# MODALITY ENCODERS
# =====================================================

class AudioEncoder(nn.Module):
    """BiLSTM over acoustic embeddings."""
    def __init__(self, d_model=256):
        super().__init__()
        self.lstm = nn.LSTM(
            d_model, d_model // 2,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return out                           # (B, T, 256)


class VideoEncoder(nn.Module):
    """2-layer Transformer encoder for visual tokens."""
    def __init__(self, d_model=256, nhead=8):
        super().__init__()
        layer    = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=2)

    def forward(self, x):
        return self.enc(x)                   # (B, T, 256)


class TextEncoder(nn.Module):
    """2-layer Transformer encoder for text tokens."""
    def __init__(self, d_model=256, nhead=8):
        super().__init__()
        layer    = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, num_layers=2)

    def forward(self, x):
        return self.enc(x)                   # (B, L, 256)


# =====================================================
# COARSE FEATURE EXTRACTION  
# =====================================================

class CoarseFeatureExtraction(nn.Module):
    
    def __init__(self, d_model=256, nhead=8):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )

    def forward(self, M_tm, M_v, M_a):
        
        attn_out, _ = self.cross_attn(query=M_tm, key=M_v, value=M_a)
        M_c = self.norm(M_tm + attn_out)
        M_c = M_c + self.ff(M_c)
        h_c = M_c.mean(dim=1)               # (B, 256)
        return M_c, h_c


# =====================================================
# DYNAMIC ATTENTION FUSION (DAF)  — coarse-to-fine
# =====================================================

class DAF(nn.Module):
  
    def __init__(self, d_model=256):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
            nn.Sigmoid()
        )

    def forward(self, fine, coarse_vec):
        """
        fine       : (B, L, d_model)
        coarse_vec : (B, d_model)
        """
        coarse_exp = coarse_vec.unsqueeze(1).expand_as(fine)  # (B, L, d_model)
        g = self.gate(torch.cat([fine, coarse_exp], dim=-1))   # (B, L, d_model)
        return g * fine + (1 - g) * coarse_exp                  # (B, L, d_model)


# =====================================================
# BERT POOL DECODER
# =====================================================

class BERTPoolDecoder(nn.Module):
   
    def __init__(self, d_model=256, num_classes=20):
        super().__init__()
        self.pool = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.Tanh()
        )
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, M_cf):
        
        cls  = M_cf[:, 0, :]           # (B, d_model)
        h    = self.pool(cls)          # (B, d_model)
        return self.classifier(h), h   # logits (B, C), pooled (B, d_model)



class PrototypeLoss(nn.Module):
   
    def __init__(self, tau=0.1):
        super().__init__()
        self.tau = tau

    def forward(self, h, y):
        
        h      = F.normalize(h, dim=-1)
        classes = y.unique()
        protos  = torch.stack([
            F.normalize(h[y == c].mean(0), dim=-1) for c in classes
        ])                                              # (C', d)

       
        label_to_idx = {c.item(): i for i, c in enumerate(classes)}
        y_idx = torch.tensor(
            [label_to_idx[yi.item()] for yi in y],
            device=h.device
        )

        logits = h @ protos.T / self.tau               # (B, C')
        return F.cross_entropy(logits, y_idx)


class InfoNCE(nn.Module):
    
    def __init__(self, tau=0.1):
        super().__init__()
        self.tau = tau

    def forward(self, anchor, positive):
        """
        anchor   : (B, d)
        positive : (B, d)
        """
        a = F.normalize(anchor,   dim=-1)
        p = F.normalize(positive, dim=-1)
        logits = a @ p.T / self.tau                    # (B, B)
        labels = torch.arange(a.size(0), device=a.device)
        return F.cross_entropy(logits, labels)


# =====================================================
# FULL MODEL
# =====================================================

class FULL_MODEL(nn.Module):


    def __init__(self, bert_path, num_classes, d_model=256):
        super().__init__()

        # --- Embedding layers ---
        self.a_embed = AcousticEmbedding(n_mels=80,   d_model=d_model)
        self.v_embed = VisualEmbedding(d_model=d_model)
        self.t_embed = TextEmbedding(bert_path,        d_model=d_model)

        # --- CTC alignment ---
        self.ctc = CTCAlign()

        # --- Modality encoders ---
        self.a_enc = AudioEncoder(d_model=d_model)
        self.v_enc = VideoEncoder(d_model=d_model)
        self.t_enc = TextEncoder(d_model=d_model)   # shared for both streams

        # --- Coarse feature extraction ---
        self.coarse = CoarseFeatureExtraction(d_model=d_model)

        # --- Two DAF modules ---
        # DAF-1: fine  → M_f  = DAF(M_tm, h_c)   used for contrastive
        # DAF-2: coarse→ M_cf = DAF(M_c,  h_c)   used for classification
        self.daf_fine   = DAF(d_model=d_model)
        self.daf_coarse = DAF(d_model=d_model)

        # --- BERT pool decoder + classifier ---
        self.decoder = BERTPoolDecoder(d_model=d_model, num_classes=num_classes)

    def forward(self, audio, video, ids_l, mask_l, ids_m, mask_m,
                a_len, v_len, t_len):

        # ---- Step 1: Embeddings ----
        Ea  = self.a_embed(audio)                  # (B, T_a, 256)
        Ev  = self.v_embed(video)                  # (B, T_v, 256)
        Etl = self.t_embed(ids_l, mask_l)          # (B, L,   256)  labeled
        Etm = self.t_embed(ids_m, mask_m)          # (B, L,   256)  masked

        # ---- Step 2: CTC alignment (aligns Ea, Ev to text length) ----
        Ea, Ev, Etl, ctc_loss = self.ctc(Ea, Ev, Etl, a_len, v_len, t_len)
        # After alignment: Ea, Ev, Etl all (B, L, 256)

        # ---- Step 3: Modality encoders ----
        Ma  = self.a_enc(Ea)                       # (B, L, 256)
        Mv  = self.v_enc(Ev)                       # (B, L, 256)
        Mtl = self.t_enc(Etl)                      # (B, L, 256)
        Mtm = self.t_enc(Etm)                      # (B, L, 256)

        # Global vectors for contrastive losses
        h_tl = Mtl.mean(dim=1)                     # (B, 256)
        h_tm = Mtm.mean(dim=1)
        h_v  = Mv.mean(dim=1)
        h_a  = Ma.mean(dim=1)

        # ---- Step 4: Coarse feature extraction (Eq. 3) ----
        # Q = Text (M_tm), K = Visual (M_v), V = Acoustic (M_a)
        M_c, h_c = self.coarse(Mtm, Mv, Ma)        # (B, L, 256), (B, 256)

        # ---- Step 5: Coarse-to-fine DAF ----
        # M_f  = DAF(fine=M_tm, coarse=h_c)  → fine features for contrastive
        # M_cf = DAF(fine=M_c,  coarse=h_c)  → coarse-enhanced for classification
        M_f  = self.daf_fine(Mtm, h_c)             # (B, L, 256)
        M_cf = self.daf_coarse(M_c, h_c)           # (B, L, 256)

        h_f = M_f.mean(dim=1)                      # (B, 256) for L_fine

        # ---- Step 6: BERT Pool Decoder → logits ----
        logits, h_pool = self.decoder(M_cf)        # (B, C), (B, 256)

        return logits, h_pool, h_tl, h_tm, h_v, h_a, h_f, ctc_loss


# =====================================================
# TRAINING SETUP
# =====================================================

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(muril_path)

dataset    = MultiModalDataset(EXCEL_PATH, AUDIO_DIR, VIDEO_DIR, tokenizer)
# =====================================================
# DATA SPLIT (70 / 15 / 15)
# =====================================================

total_size = len(dataset)

train_size = int(0.7 * total_size)
val_size   = int(0.15 * total_size)
test_size  = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    dataset,
    [train_size, val_size, test_size]
)

print("Dataset split:")
print(f"Train size: {len(train_ds)}")
print(f"Validation size: {len(val_ds)}")
print(f"Test size: {len(test_ds)}")


# DataLoaders
train_loader = DataLoader(
    train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_ds,
    batch_size=2,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=2
)
num_classes = int(dataset.df["label"].max() + 1)

model = FULL_MODEL(muril_path, num_classes).to(device)

proto_loss    = PrototypeLoss(tau=0.1)
infonce_loss  = InfoNCE(tau=0.1)


optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.2)

best_val_loss = float("inf")
patience      = 10
no_improve    = 0


# =====================================================
# TRAIN LOOP
# =====================================================

for epoch in range(100):

    model.train()
    total_loss = 0.0

    for batch in train_loader:
        logits, h_pool, h_tl, h_tm, h_v, h_a, h_f, ctc_loss = model(
            batch["audio"].to(device),
            batch["video"].to(device),
            batch["text_ids_l"].to(device),
            batch["mask_l"].to(device),
            batch["text_ids_m"].to(device),
            batch["mask_m"].to(device),
            batch["audio_len"].to(device),
            batch["video_len"].to(device),
            batch["text_len"].to(device),
        )

        y = batch["label"].to(device)

        L_cls = F.cross_entropy(logits, y)

        L_proto = proto_loss(h_pool, y)

       
        L_text = infonce_loss(h_tl, h_tm)
        L_vis  = infonce_loss(h_tl, h_v)
        L_aud  = infonce_loss(h_tl, h_a)
        L_fine = infonce_loss(h_tl, h_f)
        L_contrastive = L_text + L_vis + L_aud + L_fine

        loss = L_cls + L_proto + L_contrastive + 0.5 * ctc_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train = total_loss / len(train_loader)

    
    model.eval()
    val_loss = 0.0
    preds, trues = [], []

    with torch.no_grad():
        for batch in val_loader:
            logits, h_pool, h_tl, h_tm, h_v, h_a, h_f, ctc_loss = model(
                batch["audio"].to(device),
                batch["video"].to(device),
                batch["text_ids_l"].to(device),
                batch["mask_l"].to(device),
                batch["text_ids_m"].to(device),
                batch["mask_m"].to(device),
                batch["audio_len"].to(device),
                batch["video_len"].to(device),
                batch["text_len"].to(device),
            )

            y = batch["label"].to(device)

            L_cls         = F.cross_entropy(logits, y)
            L_proto       = proto_loss(h_pool, y)
            L_text        = infonce_loss(h_tl, h_tm)
            L_vis         = infonce_loss(h_tl, h_v)
            L_aud         = infonce_loss(h_tl, h_a)
            L_fine        = infonce_loss(h_tl, h_f)
            L_contrastive = L_text + L_vis + L_aud + L_fine
            loss          = L_cls + L_proto + L_contrastive + 0.5 * ctc_loss

            val_loss += loss.item()
            preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
            trues.extend(y.cpu().tolist())

    avg_val = val_loss / len(val_loader)

    print(
        f"Epoch {epoch+1:03d} | "
        f"Train Loss: {avg_train:.4f} | "
        f"Val Loss: {avg_val:.4f}"
    )

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        no_improve    = 0
        torch.save(model.state_dict(), "best_mvcl_daf_plusplus.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}.")
            break



model.load_state_dict(torch.load("best_mvcl_daf_plusplus.pt"))

model.eval()

preds = []
trues = []

with torch.no_grad():

    for batch in test_loader:

        logits, *_ = model(
            batch["audio"].to(device),
            batch["video"].to(device),
            batch["text_ids_l"].to(device),
            batch["mask_l"].to(device),
            batch["text_ids_m"].to(device),
            batch["mask_m"].to(device),
            batch["audio_len"].to(device),
            batch["video_len"].to(device),
            batch["text_len"].to(device),
        )

        preds.extend(torch.argmax(logits, dim=1).cpu().tolist())
        trues.extend(batch["label"].cpu().tolist())


print("\n===== Final Evaluation =====")

print(
    "Accuracy :",
    accuracy_score(trues, preds)
)

print(
    "Precision:",
    precision_score(
        trues,
        preds,
        average="weighted",
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        trues,
        preds,
        average="weighted",
        zero_division=0
    )
)

print(
    "F1 score:",
    f1_score(
        trues,
        preds,
        average="weighted",
        zero_division=0
    )
)

The tokenizer you are loading from '/kaggle/input/datasets/zzy990106/murilbasecased' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/datasets/zzy990106/murilbasecased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy 

Epoch 001 | Train Loss: 58.6089 | Val Loss: 42.2255


/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().cl

...

Epoch 051 | Train Loss: 17.7158 | Val Loss: 16.6448


/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().cl

Epoch 052 | Train Loss: 17.6389 | Val Loss: 16.8265
Early stopping at epoch 52.


/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),
/tmp/ipykernel_136/3107865678.py:157: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "text_len":   torch.tensor(mask_l.sum()),



===== Final Evaluation =====
Accuracy  : 0.2800
Precision : 0.3347
Recall    : 0.2800
F1 Score  : 0.2791
